## Generate refernce answer(ground truth) from (anchor, positive/context) data from chunk files

In [ ]:
import pyarrow
import os
import pandas as pd
import numpy as np
import hashlib

import ast
import json
import regex as re
from pprint import pprint

import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer

from google.colab import userdata
from huggingface_hub import HfApi
from datasets import load_dataset, Dataset

from time import perf_counter

import json
import pandas as pd
import os

In [ ]:
df_anchorPositive =  load_dataset('vab46/Clinical_trials_anchor-positive-pairs_EmbeddingModel-data_final').get('train').to_pandas()
df_anchorPositive.head()

README.md:   0%|          | 0.00/3.72k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.03MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7866 [00:00<?, ? examples/s]

,anchor_id,nctId,chunk_type,block_no,document_id,anchor_type,anchor,positive,chunk_char_length,status
0,0,NCT07640360,Positive_chunks,1,NCT07640360_1,macro_question,What is the main goal of this clinical trial r...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,success
1,1,NCT07640360,Positive_chunks,1,NCT07640360_1,patient_profile_question,Could I qualify for this trial if I had a rece...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,success
2,2,NCT07640360,Positive_chunks,1,NCT07640360_1,operational_question,Is there an age limit for patients to particip...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,success
3,3,NCT07640360,Positive_chunks,1,NCT07640360_1,conversational_question,I had a small stroke and my doctor said I shou...,TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Gu...,1730,success
4,4,NCT07631078,Positive_chunks,1,NCT07631078_1,macro_question,What is the primary mechanism by which kTMP-en...,TITLE: kTMP-Enhanced Motor Rehabilitation for ...,750,success


**Blocks to ensure ollama is running**

In [ ]:
#!pip install ollama

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [4,685 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:5 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,287 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,706 kB]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [113 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555

In [ ]:
#!sudo apt update && sudo apt install -y pciutils
#!sudo apt-get update && sudo apt-get install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())

GPU Available: True


In [ ]:
!nohup ollama serve > ollama.log 2>&1 & #to start ollama server

In [ ]:
# Verify the background service is active
!curl http://127.0.0.1:11434

Ollama is running

**Pull model and ccheck api request**

In [ ]:
#1.Pull model
!ollama pull qwen2.5:7b-instruct

In [ ]:
!curl -i http://127.0.0.1:11434/api/tags

HTTP/1.1 200 OK
Content-Type: application/json; charset=utf-8
Date: Sun, 06 Sep 2026 18:52:05 GMT
Content-Length: 437

{"models":[{"name":"qwen2.5:7b-instruct","model":"qwen2.5:7b-instruct","modified_at":"2026-09-06T18:52:00.013125479Z","size":4683087332,"digest":"845dbda0ea48ed749caafd9e6037047aa19acfcfd82e704d7ca97d631a0b697e","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"7.6B","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":3584},"capabilities":["completion","tools"]}]}

In [ ]:
#!pgrep -af ollama
#!cat ollama.log

In [ ]:
#!ollama --version

**LLM setup over ollama server(max_workers, keep_alive), load it**

In [ ]:
# ============================================================
# 2. OLLAMA CONFIGURATION
# ============================================================

OLLAMA_URL = "http://127.0.0.1:11434/api/generate"
MODEL = "qwen2.5:7b-instruct"

NUM_RECORDS = 2000

# Start with 2 on a T4.
# Increase to 3 only after checking VRAM / throughput.
MAX_WORKERS = 3

REQUEST_TIMEOUT = 100
MAX_RETRIES =3

# Keep model loaded in VRAM(keeplive ="30m", "10m", -1)
KEEP_ALIVE = -1

REFERENCE_OPTIONS = {
    "temperature": 0.1,
    "top_p": 0.9,
    "top_k": 30,
    "repeat_penalty": 1.05,
    "num_predict": 300,
    "num_ctx": 8192
}



In [ ]:
# 2. Set your concurrency parameters (affects runtime speed, not download speed)
os.environ["OLLAMA_NUM_PARALLEL"] = f"{MAX_WORKERS}"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"

**prompt creation (both system instructon and user prompt)**

In [ ]:
REFERENCE_PROMPT = """
You are generating reference answers for clinical-trial questions.

Each batch contains multiple questions referring to the same clinical-trial
context. Answer each question independently, using the shared context for all
questions.

Rules:
1. Answer each question using ONLY information explicitly supported by the
   provided CONTEXT.
2. Do not use outside knowledge.
3. Do not infer information that is not explicitly stated in the context.
4. Do not invent eligibility criteria, outcomes, interventions, dates, numbers,
   medical facts, or other details.
5. Answer only what the specific question asks.
6. Information may be shared across answers when it directly supports the
   corresponding question, but do not copy irrelevant information from one
   answer into another.
7. Preserve important numerical values, age limits, medical conditions,
   interventions, and eligibility constraints exactly as stated.
8. If the context does not provide enough information to answer a question,
   state that the provided context does not contain enough information to
   answer that question.
9. Keep each answer concise, factual, and directly relevant.
10. Keep each answer concise but sufficiently complete to answer the question.
Do not omit relevant supporting details merely to make the answer shorter.
When multiple criteria or conditions are relevant, include them explicitly.
Keep each answer concise but sufficiently complete to answer the question.
Do not copy the entire context unless necessary to answer the question.
11. Do not mention that you are an AI or that the answer was generated.
12. Return exactly one answer for each question, in the same order as provided.
13. Return ONLY valid JSON.
14. For eligibility or qualification questions, explicitly include the relevant inclusion or exclusion
criteria and their key thresholds needed to support the answer. Avoid unnecessary criteria or repetition.
15. If the question asks whether a patient could qualify, is eligible, or meets
study criteria, distinguish between the criteria explicitly satisfied by the question
and the additional criteria that must also be satisfied according to the context.
State those additional criteria explicitly.
16. Answer each question strictly with respect to its associated CONTEXT.
Do not transfer, assume, or reuse eligibility criteria, facts, or conclusions from
semantically similar questions or from other questions in the batch. Two questions
may be similar but must be answered independently according to their respective context.


Required format:
{
  "answers": [
    "Answer to question 1",
    "Answer to question 2",
    "Answer to question 3",
    "Answer to question 4"
  ]
}
"""

**chunk level response generation**

In [ ]:
def generate_reference_answers(chunk_group):

    context = chunk_group[0]["positive"]

    questions = "\n\n".join(
        f"anchor_id={row['anchor_id']}\n{row['anchor']}"
        for row in chunk_group
    )

    prompt = (
        REFERENCE_PROMPT
        + "\n\nCONTEXT:\n"
        + context
        + "\n\nQUESTIONS:\n"
        + questions
    )

    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "keep_alive": KEEP_ALIVE,
        "format": "json",
        "options": REFERENCE_OPTIONS
    }

    for attempt in range(MAX_RETRIES):
        try:
            if(attempt ==1):#conditional num_predict increment
                payload['options']['num_predict'] =375
            elif(attempt ==2):
                payload['options']['num_predict'] =450

            response = requests.post(
                OLLAMA_URL,
                json=payload,
                timeout=REQUEST_TIMEOUT
            )
            response.raise_for_status()

            response_text =response.json()["response"]
            result = json.loads(response_text)

            answers = result["answers"]

            if len(answers) != len(chunk_group):
                raise ValueError(
                    f"Expected {len(chunk_group)} answers, got {len(answers)}"
                )

            return answers

        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                print("\n" + "=" * 60)
                print("FAILED AFTER 3 ATTEMPTS")
                print("Error:", e)
                print("Raw response:")
                print(response_text)
                print("=" * 60)

                raise RuntimeError(
                    f"Failed after {MAX_RETRIES} attempts: {e}"
                )

            time.sleep(2 ** attempt)

**Build grouped records+ ||exceution**

Build grp blocks

In [ ]:
groups = [
    (document_id, group.to_dict(orient="records"))
    for document_id, group in df_anchorPositive.groupby("document_id", sort=False)
]

print("Total records :", len(df_anchorPositive))
print("Unique chunks :", len(groups))

Total records : 7866
Unique chunks : 2000


|| exceution on samples

In [ ]:
# ============================================================
# 4. CONCURRENT EXECUTION
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
from time import perf_counter
from tqdm.auto import tqdm

N_GROUPS = 20          # Benchmark: 10 | Production: None
CHECKPOINT_FILE = "reference_answers_checkpoint.jsonl"

run_groups = groups if N_GROUPS is None else groups[:N_GROUPS]

start_time = perf_counter()

completed = {}
failed = {}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = {
        executor.submit(
            generate_reference_answers,
            chunk_group
        ): (document_id, chunk_group)
        for document_id, chunk_group in run_groups
    }

    for future in tqdm(as_completed(futures), total=len(futures), desc=f"Workers={MAX_WORKERS}"):

        document_id, chunk_group = futures[future]

        try:
            answers = future.result()

            results = []

            for row, answer in zip(chunk_group, answers):

                results.append({
                    "anchor_id": row["anchor_id"],
                    "nctId": row["nctId"],
                    "document_id": row["document_id"],
                    "anchor_type": row["anchor_type"],
                    "anchor": row["anchor"],
                    "positive": row["positive"],
                    "reference_answer": answer,
                    "status": "success"
                })

            completed[document_id] = results

        except Exception as e:
            failed[document_id] = str(e)
            print(e)

elapsed = perf_counter() - start_time

total_records = sum(len(v) for v in completed.values())

print(f"\nWorkers       : {MAX_WORKERS}")
print(f"Groups        : {len(run_groups)}")
print(f"Records       : {total_records}")
print(f"Completed     : {len(completed)}")
print(f"Failed        : {len(failed)}")
print(f"Elapsed       : {elapsed:.2f} sec")
print(f"Groups/min    : {len(completed) / elapsed * 60:.2f}")
print(f"Records/min   : {total_records / elapsed * 60:.2f}")

Workers=3:   0%|          | 0/20 [00:00<?, ?it/s]


Workers       : 3
Groups        : 20
Records       : 80
Completed     : 20
Failed        : 0
Elapsed       : 219.63 sec
Groups/min    : 5.46
Records/min   : 21.86


In [ ]:
# ------------------------------------------------------------
# SAMPLE OUTPUTS
# ------------------------------------------------------------

sample_records = [
    record
    for results in completed.values()
    for record in results
][:2]

print("\n" + "=" * 80)
print("SAMPLE GENERATED REFERENCE ANSWERS")
print("=" * 80)

for i, record in enumerate(sample_records, 1):

    print(f"\n--- Sample {i} ---")
    print("anchor_id :", record["anchor_id"])
    print("nctId     :", record["nctId"])
    print("question  :", record["anchor"])
    print("context   :", record["positive"])
    print("reference :", record["reference_answer"])
    print("status    :", record["status"])


SAMPLE GENERATED REFERENCE ANSWERS

--- Sample 1 ---
anchor_id : 0
nctId     : NCT07640360
question  : What is the main goal of this clinical trial regarding secondary prevention after a minor stroke?
context   : TITLE: WATCH-STEP : Pilot Trial: Smartwatch-Guided Secondary Prevention After Stroke Randomized Trial of Nurse-led Program With Active vs Passive Smartwatch in Minor Stroke. A Randomized Controlled Trial Evaluating a Nurse-led Secondary Prevention and Physical Activity Program Supported by Either an Active Smartwatch (Structured Feedback) or Passive Smartwatch in Patients With Minor Stroke.
SUMMARY: After a first stroke or transient ischemic attack (TIA), the risk of recurrence is high in the weeks and months following the initial event. There are several modifiable risk factors that can reduce this risk, such as blood pressure, diet, physical activity, and smoking. Many stroke patients (NIHSS \< 5) have a low daily step count during the early recovery period, despite a good 

### final execution(paralle run with ckpt save)

In [ ]:
# ============================================================
# 4. CONCURRENT EXECUTION — PRODUCTION + CHECKPOINTING
# ============================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
from time import perf_counter
from tqdm.auto import tqdm
import json
import os

# ---------- Google Drive: persistent checkpoint ----------
from google.colab import drive
drive.mount("/content/drive")

RAG3_DIR = "/content/drive/MyDrive/RAG3"
os.makedirs(RAG3_DIR, exist_ok=True)

CHECKPOINT_FILE = os.path.join(
    RAG3_DIR, "reference_answers_checkpoint.jsonl"
)

N_GROUPS = None          # Test: 20 | Production: None


# ---------- Load successfully completed groups ----------
completed_ids = set()

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                if record.get("status") == "success":
                    completed_ids.add(record["document_id"])


# ---------- Resume: skip only successful groups ----------
remaining_groups = [
    (document_id, chunk_group)
    for document_id, chunk_group in groups
    if document_id not in completed_ids
]

run_groups = (
    remaining_groups
    if N_GROUPS is None
    else remaining_groups[:N_GROUPS]
)

print(f"Total groups      : {len(groups)}")
print(f"Already completed : {len(completed_ids)}")
print(f"Remaining groups  : {len(remaining_groups)}")
print(f"Groups this run   : {len(run_groups)}")
print(f"Workers           : {MAX_WORKERS}")




Mounted at /content/drive
Total groups      : 2000
Already completed : 822
Remaining groups  : 1178
Groups this run   : 1178
Workers           : 3


In [ ]:
# ---------- Concurrent generation ----------
start_time = perf_counter()

completed = {}
failed = {}

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = {
        executor.submit(
            generate_reference_answers,
            chunk_group
        ): (document_id, chunk_group)
        for document_id, chunk_group in run_groups
    }

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc=f"Workers={MAX_WORKERS}"
    ):
        document_id, chunk_group = futures[future]

        try:
            answers = future.result()

            # Keep context for candidate FT later
            results = [
                {
                    "anchor_id": row["anchor_id"],
                    "nctId": row["nctId"],
                    "document_id": row["document_id"],
                    "anchor_type": row["anchor_type"],
                    "anchor": row["anchor"],
                    "positive": row["positive"],
                    "reference_answer": answer,
                    "status": "success"
                }
                for row, answer in zip(chunk_group, answers)
            ]

            # ---------- Immediate durable checkpoint ----------
            with open(CHECKPOINT_FILE, "a", encoding="utf-8") as f:
                for record in results:
                    f.write(
                        json.dumps(
                            record,
                            ensure_ascii=False
                        ) + "\n"
                    )
                f.flush()
                os.fsync(f.fileno())

            completed[document_id] = results
            completed_ids.add(document_id)

        except Exception as e:
            # Failed groups remain eligible for retry
            failed[document_id] = str(e)


# ---------- Summary ----------
elapsed = perf_counter() - start_time
total_records = sum(len(v) for v in completed.values())

print(f"\nWorkers                : {MAX_WORKERS}")
print(f"Groups attempted       : {len(run_groups)}")
print(f"Groups completed       : {len(completed)}")
print(f"Groups failed          : {len(failed)}")
print(f"Records saved this run : {total_records}")
print(f"Elapsed                : {elapsed:.2f} sec")

if elapsed > 0:
    print(f"Groups/min             : {len(completed) / elapsed * 60:.2f}")
    print(f"Records/min             : {total_records / elapsed * 60:.2f}")

print(f"\nCheckpoint              : {CHECKPOINT_FILE}")

Workers=3:   0%|          | 0/1178 [00:00<?, ?it/s]


FAILED AFTER 3 ATTEMPTS
Error: Unterminated string starting at: line 5 column 5 (char 184)
Raw response:
{
  "answers": [
    "The main objective of this clinical trial is to compare the vaccine response to the 2 doses of the adjuvanted herpes zoster subunit vaccine (HZ/su, " 
    ],
    "Yes, a patient with systemic lupus erythematosus (SLE) and stable immunosuppressive treatment could qualify for this study. The inclusion criteria include being ≥ 50 years old, having ≥ 4 of the 1997 ACR13 or 2012 SLICC/ACR criteria for SLE, and having a stable dose of one or more immunosuppressive treatments for at least 4 weeks. Additionally, the patient must be eligible for the indication of the adjuvanted herpes zoster subunit vaccine and understand and voluntarily sign an informed consent form including writing consent for data protection. However, the specific type and dosage of immunosuppressive treatment must also be stable for ≥ 4 weeks, and the patient must be on a corticosteroid dose of ≥ 

### final loading to xlsx(excel) file

In [ ]:
# ============================================================
# 5. FINAL XLSX — SUCCESS + FAILURE SUMMARY
# ============================================================

# ---------- Load successful records ----------
records = []

with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            if record.get("status") == "success":
                records.append(record)

executed_reference_df = pd.DataFrame(records)


# ---------- Add failed groups ----------
failed_records = [
    {
        "anchor_id": None,
        "nctId": next(
            (row["nctId"] for row in chunk_group), None
        ),
        "document_id": document_id,
        "anchor_type": "accessible from HF final embedding dataset",
        "anchor": "accessible from HF final embedding dataset",
        "positive": "accessible from HF final embedding dataset",
        "reference_answer": None,
        "status": "failed",
        "error": error
    }
    for document_id, error in failed.items()
    for _, chunk_group in [next(
        (g for g in groups if g[0] == document_id),
        (document_id, [])
    )]
]

if failed_records:
    failed_df = pd.DataFrame(failed_records)

    executed_reference_df["error"] = None

    executed_reference_df = pd.concat(
        [executed_reference_df, failed_df],
        ignore_index=True
    )


# ---------- Save final XLSX ----------
FINAL_XLSX = os.path.join(
    RAG3_DIR,
    "reference_answers_final.xlsx"
)

executed_reference_df.to_excel(
    FINAL_XLSX,
    index=False
)

size_mb = os.path.getsize(FINAL_XLSX) / (1024 ** 2)

print(f"Total rows : {len(executed_reference_df)}")
print(f"Success    : {(executed_reference_df['status'] == 'success').sum()}")
print(f"Failed     : {(executed_reference_df['status'] == 'failed').sum()}")
print(f"XLSX size  : {size_mb:.2f} MB")
print(f"\nSaved to   : {FINAL_XLSX}")

Total rows : 7849
Success    : 7843
Failed     : 6
XLSX size  : 1.93 MB

Saved to   : /content/drive/MyDrive/RAG3/reference_answers_final.xlsx


**Load the excel and save in HF**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RAG3_DIR = "/content/drive/MyDrive/RAG3"

FINAL_XLSX = os.path.join(RAG3_DIR,"reference_answers_final.xlsx")

executed_reference_df = pd.read_excel(FINAL_XLSX)

executed_reference_df.head()

Mounted at /content/drive


,anchor_id,nctId,document_id,anchor_type,anchor,positive,reference_answer,status,error
0,8.0,NCT07690982,NCT07690982_1,macro_question,What is the purpose of this clinical trial reg...,TITLE: Validity and Reliability of the PhysioM...,The purpose of this clinical trial is to inves...,success,NaN
1,9.0,NCT07690982,NCT07690982_1,patient_profile_question,Could a patient with hemiplegia caused solely ...,TITLE: Validity and Reliability of the PhysioM...,"{'can_qualify': True, 'criteria': ['Presence o...",success,NaN
2,10.0,NCT07690982,NCT07690982_1,operational_question,How old must a patient be to participate in th...,TITLE: Validity and Reliability of the PhysioM...,A patient must be older than 40 years and youn...,success,NaN
3,11.0,NCT07690982,NCT07690982_1,conversational_question,I had a stroke and I'm interested in using an ...,TITLE: Validity and Reliability of the PhysioM...,"Based on the provided context, a patient who h...",success,NaN
4,4.0,NCT07631078,NCT07631078_1,macro_question,What is the primary mechanism by which kTMP-en...,TITLE: kTMP-Enhanced Motor Rehabilitation for ...,kTMP-enhanced motor rehabilitation aims to imp...,success,NaN


remove rejection cases

In [ ]:
print(executed_reference_df['status'].value_counts())
print(executed_reference_df['reference_answer'].apply(type).value_counts())

status
success    7843
failed        6
Name: count, dtype: int64
reference_answer
<class 'str'>      7826
<class 'bool'>       10
<class 'float'>       8
<class 'int'>         5
Name: count, dtype: int64


In [ ]:
rejection_df = executed_reference_df[
    ~executed_reference_df["reference_answer"]
    .apply(lambda x: isinstance(x, str))
].copy()

print("Rejection rows:", len(rejection_df))
print(rejection_df["reference_answer"].apply(type).value_counts())

display(rejection_df[
    ["anchor_id", "nctId", "document_id", "anchor", "reference_answer", "status"]
])

rejection_df.to_excel(
    os.path.join(RAG3_DIR, "reference_answers_rejection.xlsx"),
    index=False
)

Rejection rows: 23
reference_answer
<class 'bool'>     10
<class 'float'>     8
<class 'int'>       5
Name: count, dtype: int64


,anchor_id,nctId,document_id,anchor,reference_answer,status
147,147.0,NCT07755826,NCT07755826_1,What is the primary mechanism of repotrectinib...,NaN,success
709,709.0,NCT07656792,NCT07656792_1,What is the primary mechanism of action of BI ...,NaN,success
1247,1251.0,NCT07658521,NCT07658521_1,Is the study limited to patients who are sched...,True,success
1558,1566.0,NCT07743125,NCT07743125_1,Are participants in the SURPASS-HF trial requi...,True,success
1566,1554.0,NCT07697365,NCT07697365_1,Are participants in this study required to be ...,True,success
1586,1590.0,NCT07687186,NCT07687186_1,Is the trial open to patients aged between 18 ...,True,success
2292,2300.0,NCT07730788,NCT07730788_1,Is this clinical trial open to patients aged 1...,True,success
2963,2975.0,NCT07636525,NCT07636525_1,How many times will participating families nee...,2,success
3446,3458.0,NCT07646223,NCT07646223_1,Are children participating in this trial requi...,True,success
3485,3497.0,NCT07708025,NCT07708025_1,Are participants required to have been taking ...,True,success


In [ ]:
cleaned_df = executed_reference_df.drop(
    index=rejection_df.index
).copy()

In [ ]:
cleaned_df['status'].value_counts()

,count
status,
success,7826


In [ ]:
cleaned_df.shape

(7826, 9)

**normalize the dict ref_answers to string and check. finall push to HF**

In [ ]:
dataset_clean = Dataset.from_pandas(cleaned_df)

dataset_clean.push_to_hub(f"vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA_ft")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  19%|#8        |  539kB / 2.88MB            

CommitInfo(commit_url='https://huggingface.co/datasets/vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA_ft/commit/252d8677a7dd084ad0e958100355bb73106c6dc0', commit_message='Upload dataset', commit_description='', oid='252d8677a7dd084ad0e958100355bb73106c6dc0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA_ft', endpoint='https://huggingface.co', repo_type='dataset', repo_id='vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA_ft'), pr_revision=None, pr_num=None)

**push new_files to hub(after uploading on disc)**

In [ ]:
os.listdir(os.getcwd())

['.config', 'refernce_answers_new.xlsx', 'sample_data']

In [ ]:
df_refernceAnswer = pd.read_excel('/content/refernce_answers_new.xlsx', engine='openpyxl')

dataset_clean = Dataset.from_pandas(df_refernceAnswer)

dataset_clean.push_to_hub(f"vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA-junk_handled_ft")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmp4duhp4gx.parquet    :  18%|#7        |  586kB / 3.32MB            

CommitInfo(commit_url='https://huggingface.co/datasets/vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA-junk_handled_ft/commit/31bc9cb227d2cd5c3ad39dee755805c29322fd18', commit_message='Upload dataset', commit_description='', oid='31bc9cb227d2cd5c3ad39dee755805c29322fd18', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA-junk_handled_ft', endpoint='https://huggingface.co', repo_type='dataset', repo_id='vab46/Clinical_trials_anchor-contextORpositive-ground-truth_LLM_LORA-junk_handled_ft'), pr_revision=None, pr_num=None)